# Porovnání přeškálování: $R\,\psi$ vs. $\chi\,\psi$ se $\chi = \sqrt{1+R^2}$

Vezmeme **jedno** exaktní odcházející řešení 3D vlnové rovnice

$$
\psi(T, R) = \frac{f(T-R) - f(T+R)}{R}, \qquad f(u) = e^{-(u + u_0)^2/\sigma_u^2},
$$

a v hyperboloidálních souřadnicích $(t, r)$ (stejná transformace jako všude:
$h = \sqrt{S^2+R^2}-S$, $R = r/\Omega$, $\Omega = 1 - r^2/S^2$) porovnáme jeho dvě přeškálované verze:

- $R\,\psi$ — proměnná z `HypSlc_WESystemNumSol.ipynb` (lichá v počátku, splňuje čistou 1D vlnovou rovnici),
- $\tilde\psi = \chi\,\psi$, $\chi = \sqrt{1+R^2}$ — proměnná z `HypSlc_WESystemNumSol_chi.ipynb`
  po vzoru Gautam et al. (sudá a $\simeq \psi$ u počátku).

Obě se liší jen faktorem
$$
\frac{\chi}{R} = \frac{\sqrt{1+R^2}}{R} = \frac{\sqrt{r^2 + \Omega^2}}{r},
$$
který jde od $\infty$ v počátku (kde $R\psi \to 0$, ale $\chi\psi \to \psi$) k **přesně 1 na scri** —
tam jsou obě proměnné totožné a rovné radiačnímu signálu $f(t - S)$.

Parametry volíme stejné jako v numerických notebookách ($S = 1$, balík na $r_0 = 0.4$,
tj. $R_0^{\rm fyz} \approx 0.476$): poměr $\chi/R$ v místě balíku je $\approx 2.33$, což přesně
vysvětluje, proč numerika s $\tilde\psi$ (start s amplitudou 1) dávala signál $\approx 0.43$,
zatímco numerika s $R\psi$ dávala 1.

In [ ]:
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
# --- Parametry (stejné jako v numerických notebookách) ---
S = 1.0
r0 = 0.4        # střed balíku v kompaktifikované souřadnici na řezu t = 0
sigma_u = 0.05  # šířka balíku v retardovaném čase


# --- Kompaktifikace a výšková funkce ---
def compact_helpers(r):
    Omega = 1 - (r / S) ** 2
    Q = np.sqrt(S**2 * Omega**2 + r**2)
    R_of_r = np.divide(r, Omega, out=np.full_like(r, np.inf), where=Omega > 0)
    k_r = S - S**2 * Omega / (r + Q)  # k = R - h(R)
    v_geom = np.divide(r + Q, Omega, out=np.full_like(r, np.inf), where=Omega > 0) - S
    ratio = np.ones_like(r)           # chi/R = sqrt(r^2 + Omega^2)/r, na scri přesně 1
    ratio[1:] = np.sqrt(r[1:] ** 2 + Omega[1:] ** 2) / r[1:]
    return Omega, Q, R_of_r, k_r, v_geom, ratio


# střed balíku v u: aby na řezu t = 0 seděl v r = r0
_, _, _, k0, _, _ = compact_helpers(np.array([0.0, r0]))
u0 = k0[1]
print(f"u0 = k(r0) = {u0:.4f}")


def f(u):
    return np.exp(-np.power((u + u0) / sigma_u, 2))


def fprime(u):
    return -2 * (u + u0) / sigma_u**2 * f(u)


# --- Tři podoby téhož řešení v (t, r) ---
def Rpsi_tr(t, k_r, v_geom):
    """R psi = f(u) - f(v), lichá v R, na scri = radiační signál."""
    return f(t - k_r) - f(t + v_geom)


def psi_tr(t, R_of_r, k_r, v_geom):
    """Nepřeškálované psi, v počátku l'Hospital: psi(t, 0) = -2 f'(t)."""
    chi = Rpsi_tr(t, k_r, v_geom)
    R_safe = np.where((R_of_r > 1e-12) & np.isfinite(R_of_r), R_of_r, 1.0)
    psi = np.where(np.isfinite(R_of_r), chi / R_safe, 0.0)
    return np.where(R_of_r > 1e-12, psi, -2 * fprime(t))


def chipsi_tr(t, k_r, v_geom, ratio, R_of_r):
    """chi psi = (chi/R) * (R psi); v počátku chi = 1, tedy chi psi -> psi(0) = -2 f'(t)."""
    val = ratio * Rpsi_tr(t, k_r, v_geom)
    return np.where(R_of_r > 1e-12, val, -2 * fprime(t))

## Škálovací funkce

Vlevo obě škálovací funkce ve fyzikálním $R$: $\chi = \sqrt{1+R^2}$ se od $R$ liší jen pro
$R \lesssim 2$ a u počátku jde k 1 (zachová paritu a řád $\psi$). Vpravo jejich poměr $\chi/R$
na kompaktifikované mřížce — vyznačené místo startu balíku ukazuje faktor $\approx 2.33$.

In [ ]:
r_grid = np.linspace(0.0, S, 1001)
Omega_g, Q_g, R_of_r_g, k_g, v_geom_g, ratio_g = compact_helpers(r_grid)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

R_line = np.linspace(0, 8, 400)
ax1.plot(R_line, R_line, lw=2, color="firebrick", label="$R$")
ax1.plot(R_line, np.sqrt(1 + R_line**2), lw=2, color="steelblue", label="$\\chi = \\sqrt{1+R^2}$")
ax1.set_xlabel("$R$ (fyzikální)")
ax1.set_title("škálovací funkce")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.semilogy(r_grid[1:], ratio_g[1:], lw=2, color="black")
i0 = np.argmin(np.abs(r_grid - r0))
ax2.plot(r0, ratio_g[i0], "o", color="firebrick")
ax2.annotate(f"start balíku: $\\chi/R \\approx {ratio_g[i0]:.2f}$", (r0, ratio_g[i0]),
             textcoords="offset points", xytext=(15, 10))
ax2.axhline(1, color="gray", lw=0.8)
ax2.set_xlabel("$r$ (kompaktifikované)")
ax2.set_title("poměr $\\chi/R$ (na scri přesně 1)")
ax2.grid(True, linestyle="--", alpha=0.6)
plt.show()

## Prostoročasové diagramy v $(r, t)$

Totéž řešení třikrát: nepřeškálované $\psi$ (pro kontext), $R\psi$ a $\chi\psi$.
U počátku se $\chi\psi$ chová jako $\psi$ (sudé, plná amplituda), u scri jako $R\psi$.

In [ ]:
nT, nx = 600, 600
t_grid = np.linspace(0, 2, nT)
tc = t_grid[:, None]

data_psi = psi_tr(tc, R_of_r_g[None, :], k_g[None, :], v_geom_g[None, :])
data_Rpsi = Rpsi_tr(tc, k_g[None, :], v_geom_g[None, :])
data_chipsi = chipsi_tr(tc, k_g[None, :], v_geom_g[None, :], ratio_g[None, :], R_of_r_g[None, :])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, data, title, vmax in [
    (axes[0], data_psi, "$\\psi(t, r)$ — nepřeškálované", 2.2),
    (axes[1], data_Rpsi, "$R\\psi(t, r)$", 1.0),
    (axes[2], data_chipsi, "$\\chi\\psi(t, r)$", 2.4),
]:
    im = ax.pcolormesh(r_grid, t_grid, data, cmap="viridis", vmin=0, vmax=vmax, rasterized=True)
    ax.set_xlabel("$r$")
    ax.set_title(title)
    ax.axvline(S, color="w", lw=1.5, linestyle=":")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
axes[0].set_ylabel("$t$")
plt.tight_layout()
plt.show()

## Animace: všechny tři proměnné najednou

Šedě $\psi$, červeně $R\psi$, modře $\chi\psi$. U počátku modrá kopíruje šedou
(a červená tam umírá k nule — lichá parita), u scri modrá splyne s červenou
a obě dorazí se stejným signálem.

In [ ]:
r_anim = np.linspace(0, S, 1001)
Omega_a, Q_a, R_of_r_a, k_a, v_geom_a, ratio_a = compact_helpers(r_anim)
t_frames = np.linspace(0, 2, 81)

fig, ax = plt.subplots(figsize=(10, 5))
(l_psi,) = ax.plot(r_anim, psi_tr(t_frames[0], R_of_r_a, k_a, v_geom_a), lw=1.5, color="gray",
                   alpha=0.8, label="$\\psi$")
(l_R,) = ax.plot(r_anim, Rpsi_tr(t_frames[0], k_a, v_geom_a), lw=2, color="firebrick",
                 label="$R\\psi$")
(l_chi,) = ax.plot(r_anim, chipsi_tr(t_frames[0], k_a, v_geom_a, ratio_a, R_of_r_a), lw=2,
                   color="steelblue", label="$\\chi\\psi$")

ax.axvline(S, color="black", lw=1.5, linestyle=":")
ax.set_xlim(0, S)
ax.set_ylim(-0.2, 2.6)
ax.set_xlabel("$r$ (kompaktifikované)")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.6)
title = ax.set_title("")


def update(frame):
    t = t_frames[frame]
    l_psi.set_ydata(psi_tr(t, R_of_r_a, k_a, v_geom_a))
    l_R.set_ydata(Rpsi_tr(t, k_a, v_geom_a))
    l_chi.set_ydata(chipsi_tr(t, k_a, v_geom_a, ratio_a, R_of_r_a))
    title.set_text(f"$t = {t:.2f}$")
    return (l_psi, l_R, l_chi, title)


ani = FuncAnimation(fig, update, frames=len(t_frames), interval=75, blit=False, cache_frame_data=False)
plt.close()
HTML(ani.to_jshtml())

## Signál na $\mathcal{I}^+$

Na scri je $\chi/R = 1$ přesně, takže obě přeškálované proměnné nesou **identický** radiační
signál $f(t - S)$ — volba škálovací funkce mění chování v počátku a v mezilehlé oblasti,
ale to, co čteme na nulovém nekonečnu, je stejné.

In [ ]:
t_sig = np.linspace(0, 2, 2001)
sig_R = Rpsi_tr(t_sig, k_g[-1], v_geom_g[-1])
sig_chi = ratio_g[-1] * sig_R

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_sig, sig_R, lw=3, color="firebrick", label="$R\\psi$ na scri")
ax.plot(t_sig, sig_chi, lw=1.5, color="steelblue", linestyle="--", label="$\\chi\\psi$ na scri")
ax.set_xlabel("$t$")
ax.set_ylabel("signál")
ax.set_title("Signál na $\\mathcal{I}^+$: obě přeškálování dávají totéž")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)
plt.show()

print(f"max |rozdíl| obou signálů na scri: {np.abs(sig_R - sig_chi).max():.3e}")
print(f"maximum signálu v čase t = {t_sig[np.argmax(np.abs(sig_R))]:.3f}")

## Shrnutí

| | $R\psi$ | $\chi\psi$, $\chi = \sqrt{1+R^2}$ |
|---|---|---|
| parita v počátku | lichá (nulová v $r=0$) | sudá, $\chi\psi \simeq \psi$ |
| rovnice | čistá 1D vlnová (žádné extra členy) | extra regulární členy $a$, $b$ |
| amplituda podél paprsku | konstantní ($= f(u)$) | klesá faktorem $\chi/R \to 1$ |
| signál na scri | $f(t-S)$ | $f(t-S)$ — **totožný** |

Numerické protějšky: `HypSlc_WESystemNumSol.ipynb` ($R\psi$) a `HypSlc_WESystemNumSol_chi.ipynb` ($\chi\psi$);
exaktní vizualizace mezikroků v `HypSlc_WESolVizualization.ipynb`.